In [1]:
# =====================================================================
# CELL 2: IMPORT LIBRARY & LOAD MODEL (LOCAL SAVE WHISPER)
# =====================================================================
import os
import time
import joblib
import numpy as np
import scipy.io.wavfile as wav
import sounddevice as sd
import torch
import re
import pandas as pd
from faster_whisper import WhisperModel, download_model

# 1. LOAD PIPELINE MODEL CLASSIFICATION ARYA (6 TARGET)
MODEL_NLP_PATH = "../../../models/ticketing/model_tfidf_rf_exigen.pkl"

print("⏳ Memuat model klasifikasi tiket pintar Exigen...")
if os.path.exists(MODEL_NLP_PATH):
    pipeline_nlp = joblib.load(MODEL_NLP_PATH)
    print("✅ Model NLP berhasil dimuat!")
else:
    raise FileNotFoundError(f"⚠️ Model tidak ditemukan di jalur: {MODEL_NLP_PATH}")

# 2. INISIALISASI & SAVE MODEL WHISPER LARGE SECARA LOKAL
# Kita buatkan sub-folder agar rapi, karena Whisper berisi beberapa file (.bin, .json, dll)
WHISPER_LOCAL_DIR = "../../../models/whisper-large-v3"
DEVICE_HARDWARE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n⏳ Mengecek ketersediaan model Whisper di folder lokal: {WHISPER_LOCAL_DIR}")

# Fitur ini otomatis mengecek: jika model belum ada, dia akan mendownload.
# Jika sudah ada di dalam folder tersebut, dia akan memotong proses download!
path_model_lokal = download_model("large-v3", output_dir=WHISPER_LOCAL_DIR)
print("✅ File model Whisper terkonfirmasi ada di sistem lokal!")

print(f"⏳ Memuat Model Whisper ke memori (Varian: LARGE, Perangkat: {DEVICE_HARDWARE})...")

# Kita paksa Whisper untuk membaca dari folder lokal (path_model_lokal)
# Gunakan "int8_float16" atau "int8" agar VRAM tetap hemat
model_stt = WhisperModel(path_model_lokal, device=DEVICE_HARDWARE, compute_type="int8_float16")

print("✅ Model Whisper Large siap tempur 100% Offline!")

⏳ Memuat model klasifikasi tiket pintar Exigen...
✅ Model NLP berhasil dimuat!

⏳ Mengecek ketersediaan model Whisper di folder lokal: ../../../models/whisper-large-v3


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\huggingface_hub\file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

✅ File model Whisper terkonfirmasi ada di sistem lokal!
⏳ Memuat Model Whisper ke memori (Varian: LARGE, Perangkat: cuda)...
✅ Model Whisper Large siap tempur 100% Offline!


In [ ]:
# =====================================================================
# CELL 3: PREPROCESSING & KAMUS SLANG
# =====================================================================
print("⏳ Mengunduh Kamus Bahasa Gaul (Slang Dictionary)...")
url_slang = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
try:
    df_slang = pd.read_csv(url_slang)
    slang_dict_external = dict(zip(df_slang['slang'], df_slang['formal']))
    print(f"✅ Berhasil memuat {len(slang_dict_external)} kata slang/gaul!")
except Exception as e:
    print(f"⚠️ Gagal mengunduh kamus slang. Error: {e}")
    slang_dict_external = {} 

def super_clean_text(text):
    text = text.lower()
    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text) # Hapus typo stuttering
    text = re.sub(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\s*([a-z0-9]+)\b', r'gedung_\2', text)
    text = re.sub(r'\b(lantai|lt\.?|level)\s*([a-z0-9]+)\b', r'lantai_\2', text)
    text = re.sub(r'\b(ruang|rg\.?|kamar|kmr)\s*([a-z0-9]+)\b', r'ruang_\2', text)
    text = re.sub(r'[^a-z0-9_]', ' ', text).strip()
    
    kata_kata = text.split()
    kata_normal = [slang_dict_external.get(kata, kata) for kata in kata_kata]
    return " ".join(kata_normal)

⏳ Mengunduh Kamus Bahasa Gaul (Slang Dictionary)...
✅ Berhasil memuat 4331 kata slang/gaul!


In [3]:
# =====================================================================
# CELL 4: FUNGSI AUDIO STREAMING & TRANSKRIPSI (STABILIZED)
# =====================================================================

def rekam_suara_mikrofon(nama_file="laporan_suara_temp.wav", durasi_maksimal=7, samplerate=16000):
    print(f"\n🎙️ [AUDIO] Mikrofon AKTIF... Silakan bicara (Maksimal {durasi_maksimal} detik).")
    print("🔴 PEREKAMAN DIMULAI...")
    
    # Kadang int16 di sounddevice butuh konversi spesifik, tapi scipy.io.wavfile menanganinya dengan baik
    rekaman = sd.rec(int(durasi_maksimal * samplerate), samplerate=samplerate, channels=1, dtype="int16")
    sd.wait() 
    
    print("🟢 Perekaman selesai.")
    wav.write(nama_file, samplerate, rekaman)
    return nama_file

def pipeline_speech_to_text(audio_path):
    if not os.path.exists(audio_path): 
        return ""
        
    start_time = time.time()
    
    # Penyesuaian untuk model Large: condition_on_previous_text=False mencegah halusinasi berulang
    segments, info = model_stt.transcribe(
        audio_path, 
        language="id", 
        beam_size=5, 
        condition_on_previous_text=False,
        vad_filter=True, 
        vad_parameters=dict(min_silence_duration_ms=500)
    )
    
    hasil_teks = [segment.text for segment in segments]
    duration = time.time() - start_time
    
    print(f"⏱️ Deteksi Bahasa: {info.language} (Akurasi: {info.language_probability:.2%})")
    print(f"⏱️ Waktu STT     : {duration:.2f} detik")
    
    return " ".join(hasil_teks).strip()

In [6]:
# =====================================================================
# CELL 5: EKSEKUSI PIPELINE (VOICE -> TEXT -> PREDICT TARGET)
# =====================================================================

# 1. Mulai Merekam Suara Anda
file_audio = rekam_suara_mikrofon(durasi_maksimal=7)

# 2. Transkripsi Audio ke Teks Mentah
print("\n🔀 Memulai proses transkripsi teks...")
teks_mentah = pipeline_speech_to_text(file_audio)
print(f"💬 Teks Mentah : \"{teks_mentah}\"")

# 3. Prediksi Rute Tiket Fase 1
if teks_mentah:
    teks_bersih = super_clean_text(teks_mentah)
    print(f"✨ Teks Bersih : \"{teks_bersih}\"")
    
    print("\n🤖 Meminta Model Multi-Output memprediksi...")
    tebakan_ai = pipeline_nlp.predict([teks_bersih])
    
    print("\n" + "=" * 50)
    print("🎯 TEBAKAN OTOMATIS TIKET PINTAR FASE 1 (STATUS: OPEN)")
    print("=" * 50)
    print(f" 🔹 Tipe Aset          : {tebakan_ai[0][0]}")
    print(f" 🔹 Lokasi Gedung      : {tebakan_ai[0][1]}")
    print(f" 🔹 Lokasi Lantai      : {tebakan_ai[0][2]}")
    print(f" 🔹 Lokasi Zona        : {tebakan_ai[0][3]}")
    print(f" 🔹 Kategori Dept      : {tebakan_ai[0][4]}")
    print(f" 🔹 Severity (Awal)    : {tebakan_ai[0][5]}")
    print("=" * 50)
    
    if tebakan_ai[0][5] in ["Berat", "Fatal", "Tinggi"]:
        print("🚨 ALERT: Severity TINGGI! Memicu WhatsApp API ke teknisi.")
else:
    print("❌ Suara tidak terdeteksi jelas.")


🎙️ [AUDIO] Mikrofon AKTIF... Silakan bicara (Maksimal 7 detik).
🔴 PEREKAMAN DIMULAI...
🟢 Perekaman selesai.

🔀 Memulai proses transkripsi teks...
⏱️ Deteksi Bahasa: id (Akurasi: 100.00%)
⏱️ Waktu STT     : 1.33 detik
💬 Teks Mentah : "Hai tolong cok  Gue kejebak di lift  Rantai 2"
✨ Teks Bersih : "hai tolong cok gue kejebak di lift rantai 2"

🤖 Meminta Model Multi-Output memprediksi...

🎯 TEBAKAN OTOMATIS TIKET PINTAR FASE 1 (STATUS: OPEN)
 🔹 Tipe Aset          : Lift Executive
 🔹 Lokasi Gedung      : Gedung Utama
 🔹 Lokasi Lantai      : 3
 🔹 Lokasi Zona        : Tengah
 🔹 Kategori Dept      : Sistem Transportasi Gedung
 🔹 Severity (Awal)    : Sedang
